**`harmonize_buildings`**

Create a spine for buildings (identifiers for all observed buildings)

The current version is specific to the U.S. context (not generalized)

# Configure

In [ ]:
import argparse
import warnings

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from openplaces.api import get_admin, read_entities
from openplaces.geo.ids import (
    add_openlocationcode_index,
)
from openplaces.geo.overlay import overlay_polygons
from openplaces.geo.polygon import (
    get_areas,
    get_intersection_over_union,
    resolve_overlapping_polygons,
)
from openplaces.io import save_parquet, share
from openplaces.io.transform import make_index_unique, remap
from openplaces.path import external_path, path, share_path
from openplaces.recipe import get_output_path
from openplaces.viz import show_building

In [ ]:
# Define arguments
parser = argparse.ArgumentParser(
    description='Harmonize building data from multiple recipes'
)
parser.add_argument(
    '--admin_ids',
    help='Administrative unit IDs to ingest (e.g., "US-RI")',
    nargs='*',
)
parser.add_argument(
    '--show_examples',
    help='Make maps of examples (using matplotlib)',
    action='store_true',
)
parser.add_argument(
    '--verbose',
    help='If True, print outputs while processing data',
    action='store_true',
)

# Test arguments

In [ ]:
ARGS_TEST = (
    # Example 1: Brunswick, NC (hurricane risk case)
    '--admin_ids US-NC-BS '
    # '--admin_ids US-NC-BL '
    # '--admin_ids US-NC-CU '  # Duplicate building ID
    # Example 2: Buncombe, NC (fluvial flood risk case)
    # '--admin_ids US-NC-BO'
    # Example 3: Jefferson and Harris, NC (hurricane risk cases)
    # '--admin_ids US-TX-JE US-TX-RR'
    '--show_examples '
    '--verbose'
)

# Convert argument string to list of strings
args_list = [x for x in ARGS_TEST.split(' ') if x != '']

# Parse list of arguments
args = parser.parse_args(args_list)

# Display arguments to check if parsing worked as expected
args

# Prepare data

In [ ]:
# Future start of loop
for admin_id in args.admin_ids:
    break

print(admin_id)

# For visualization purposes
admin3 = get_admin(admin_id, level=3, geom=True)
admin3

## Prepare buildings

In [ ]:
buildings_obm = read_entities('building-obm-2025', admin_id, geom=True)
buildings_microsoft = read_entities('US_building-microsoft-v2', admin_id, geom=True)
buildings_fema = read_entities('US_building-fema-2023', admin_id, geom=True)
buildings_nsi = read_entities('US_building-nsi-2022', admin_id, geom=True)
if admin_id.startswith('US-NC'):
    buildings_nc = read_entities('US-NC_building-ncdps-2023', admin_id, geom=True)

In [ ]:
# Hesam Soleimani's first inventory
BUILDINGS_CHEER_PATH = external_path(
    'US-NC', 'building-cheer-v0', filename='Inventory_v0_NC.parquet'
)
# Get county FIPS code
county_fips = get_admin(admin_id)['admin3_id_admin1'].values[0]
buildings_cheer = gpd.read_parquet(
    BUILDINGS_CHEER_PATH, filters=[('county', '==', county_fips)]
).set_index('bid')

In [ ]:
# Remap building groups
buildings_nsi = remap(buildings_nsi, 'US_building-nsi-2022_purpose-subgroup-remap')
buildings_fema = remap(buildings_fema, 'US_building-fema-2023_purpose-subgroup-remap')

In [ ]:
# Calculate polygon areas
buildings_microsoft['m2'] = get_areas(buildings_microsoft, 'm2')
buildings_fema['m2'] = get_areas(buildings_fema, 'm2')

In [ ]:
# Remove overlapping polygons in OBM if they exist
buildings_obm = resolve_overlapping_polygons(buildings_obm, keep=False)

## Prepare parcels

In [ ]:
parcels = read_entities('US-NC_parcel-nconemap-2025', admin_id, geom=True)

# US-NC: remove parcels that are identical in everything but the
# `source_deed` these seem to be duplicates from records going over
# multiple pages
mask_duplicates = parcels.drop(columns=['source_deed']).duplicated()
unique_parcels = parcels[~mask_duplicates].copy()

# Identify identical polygons
unique_parcels['has_duplicate_geometry'] = unique_parcels['geo_id'].duplicated(
    keep=False
)
if unique_parcels['has_duplicate_geometry'].any():
    warnings.warn('Duplicate `geo_id` values found after dropping duplicate parcels.')
unique_parcel_polygons = unique_parcels[~unique_parcels['geo_id'].duplicated()][
    ['geometry', 'geo_id', 'has_duplicate_geometry']
]
# Use 'geo_id' as index
unique_parcel_polygons.index = unique_parcel_polygons['geo_id'].rename('parcel_id')

# Join key indicators
unique_parcels_aggregated_to_polygon = unique_parcels.groupby('geo_id').agg(
    {'purpose_group': ', '.join, 'improvement_value': 'mean'}
)
unique_parcel_polygons = unique_parcel_polygons.join(
    unique_parcels_aggregated_to_polygon
)
unique_parcel_polygons['ha'] = get_areas(unique_parcel_polygons, 'ha')

unique_parcel_polygons['improvement_value_per_ha'] = (
    unique_parcel_polygons['improvement_value'] / unique_parcel_polygons['ha']
)

In [ ]:
if args.verbose:
    print(f'{len(buildings_obm):,d} footprints by OpenBuildingMap')
    print(f'{len(buildings_microsoft):,d} footprints by Microsoft')
    print(f'{len(buildings_fema):,d} footprints by FEMA')
    if admin_id.startswith('US-NC'):
        print(f'{len(buildings_nc):,d} dwelling footprints by NC-DPS')
    print(f'{len(buildings_nsi):,d} dwellings by NSI (points)')
    print(f'{len(buildings_cheer):,d} buildings in V0 inventory (CHEER)')
    print(f'{len(parcels):,d} parcels')

# Create building spine
1. Keep OpenBuildingMap footprints
2. Add Microsoft footprints that don't overlap
3. Add FEMA footprints that don't overlap
4. Add NSI points located outside of footprints
5. Add parcel-level information

In [ ]:
FOOTPRINT_OVERLAP_IOU_MAX = 0.02

## Keep OpenBuildingMap footprints

In [ ]:
buildings_obm['source'] = 'obm'
buildings = buildings_obm[['geometry', 'source']]

## Add Microsoft footprints that don't overlap
Allow only minor overlaps (geolocation errors)

In [ ]:
buildings_obm_microsoft = overlay_polygons(
    get_output_path('building-obm-2025', admin_id),
    get_output_path('US_building-microsoft-v2', admin_id),
    suffixes=('_obm', '_microsoft'),
    iou=True,
).sort_values('iou', ascending=False)
buildings_obm_microsoft

In [ ]:
# Accepted size of overlap understood as identifying separate buildings
building_microsoft_ids = buildings_obm_microsoft[
    buildings_obm_microsoft['iou'].gt(FOOTPRINT_OVERLAP_IOU_MAX)
].index.get_level_values('footprint_id_microsoft')

buildings_microsoft_non_overlapping = buildings_microsoft[
    ~buildings_microsoft.index.isin(building_microsoft_ids)
]
buildings_microsoft_non_overlapping['source'] = 'microsoft'

buildings = pd.concat(
    [buildings, buildings_microsoft_non_overlapping[['geometry', 'source']]]
).sort_index()

## Add FEMA footprints that don't overlap

In [ ]:
# Using get_intersection_over_union so we don't have to save to disc
buildings_hybrid_fema = get_intersection_over_union(
    buildings,
    buildings_fema,
    suffixes=('hybrid', 'fema'),
).sort_values('iou', ascending=False)
buildings_hybrid_fema

In [ ]:
# Accepted size of overlap understood as identifying separate buildings
building_fema_ids = buildings_hybrid_fema[
    buildings_hybrid_fema['iou'].gt(FOOTPRINT_OVERLAP_IOU_MAX)
].index.get_level_values('footprint_id_fema')

buildings_fema_non_overlapping = buildings_fema[
    ~buildings_fema.index.isin(building_fema_ids)
]
buildings_fema_non_overlapping['source'] = 'fema'

buildings = pd.concat(
    [buildings, buildings_fema_non_overlapping[['geometry', 'source']]]
).sort_index()

buildings['source'] = pd.Categorical(
    buildings['source'], categories=['obm', 'microsoft', 'fema'], ordered=True
)

## Add buildings from parcels
### Identify parcels with footprints

This is now the slowest step with 30 sec in UN-NC-BS

In [ ]:
# Columns to keep in harmonized dataset
FOOTPRINT_PARCEL_COLS = [
    'footprint_id',
    'parcel_id',
    'm2_intersection',
    'iou',
    'm2_intersection_inner',
    'fraction_of_largest',
]

FOOTPRINT_PARCEL_M2_INTERSECTION_MIN = 10

FOOTPRINT_PARCEL_MIN_FRACTION_OF_LARGEST = 1 / 6

In [ ]:
footprints_on_parcels = get_intersection_over_union(
    buildings,
    unique_parcel_polygons,
    suffixes=('building', 'parcel'),
    how='identity',
    drop_geometries=False,
)
footprints_on_parcels[['m2_intersection', 'iou', 'geometry']]

### Keep unique parcels per footprint

In [ ]:
mask_multiparcel_footprints = footprints_on_parcels.index.get_level_values(
    'footprint_id'
).duplicated(keep=False)

cols = [v for v in FOOTPRINT_PARCEL_COLS if v in footprints_on_parcels]

# Initiate crosswalk (footprint_id > parcel_id)
footprints_unique_parcel = (
    footprints_on_parcels[~mask_multiparcel_footprints]
    .reset_index()
    .set_index('footprint_id')[['parcel_id'] + cols]
)
link = np.where(
    footprints_unique_parcel['parcel_id'].notnull(), 'unique parcel', 'no parcel'
)
footprints_unique_parcel.insert(1, 'link', link)
# footprints_unique_parcel

### Add unique parcels per footprint after removing small neighbors

In [ ]:
# Compute inner buffer (to identify slivers)
multiparcel_footprints = footprints_on_parcels[mask_multiparcel_footprints].copy()
multiparcel_footprints['fraction_of_largest'] = (
    multiparcel_footprints.groupby('footprint_id', group_keys=False)['m2_intersection']
    .apply(lambda x: (x / x.max()))
    .round(3)
)
multiparcel_footprints_without_small_neighbor = multiparcel_footprints.query(
    f'fraction_of_largest >= {FOOTPRINT_PARCEL_MIN_FRACTION_OF_LARGEST} '
    f'and m2_intersection >= {FOOTPRINT_PARCEL_M2_INTERSECTION_MIN}'
)
# multiparcel_footprints[['iou', 'm2_intersection', 'fraction_of_largest']]

In [ ]:
mask_multiparcel_footprints_to_split = (
    multiparcel_footprints_without_small_neighbor.index.get_level_values(
        'footprint_id'
    ).duplicated(keep=False)
)

cols = [v for v in FOOTPRINT_PARCEL_COLS if v in multiparcel_footprints]

# Add newly resolved unique parcels to crosswalk
footprints_unique_parcel_without_small_neighbor = (
    multiparcel_footprints_without_small_neighbor[~mask_multiparcel_footprints_to_split]
    .reset_index()
    .set_index('footprint_id')[['parcel_id'] + cols]
)
link = np.where(
    footprints_unique_parcel_without_small_neighbor['parcel_id'].notnull(),
    'unique parcel',
    'no parcel',
)
footprints_unique_parcel_without_small_neighbor.insert(
    1, 'link', link + ' (dropping small neighbor)'
)
footprints_unique_parcel = pd.concat(
    [
        footprints_unique_parcel.drop(
            set(footprints_unique_parcel.index)
            & set(footprints_unique_parcel_without_small_neighbor.index)
        ),
        footprints_unique_parcel_without_small_neighbor,
    ]
).sort_index()

# footprints_unique_parcel

### Assign remaining footprints to multiple parcels

In [ ]:
cols = [
    v
    for v in FOOTPRINT_PARCEL_COLS
    if v in multiparcel_footprints_without_small_neighbor
]

multiparcel_footprints_to_split = multiparcel_footprints_without_small_neighbor[
    mask_multiparcel_footprints_to_split
][cols + ['geometry']]
multiparcel_footprints_to_split.insert(0, 'link', 'multi-parcel footprint')
# multiparcel_footprints_to_split.sort_index()

In [ ]:
footprint_parcels = pd.concat(
    [
        footprints_unique_parcel.reset_index().set_index(['footprint_id', 'parcel_id']),
        multiparcel_footprints_to_split.drop(columns='geometry'),
    ]
).sort_index()

print(footprint_parcels['link'].value_counts())

footprint_parcels

In [ ]:
if args.show_examples:
    # FOOTPRINT_ID = '8753WXJF+4Q4'  # manufactured home that has been moved
    # footprint_id = FOOTPRINT_ID

    # Display multi-parcel footprint
    footprint_id = multiparcel_footprints_to_split.sample().index[0][0]
    print('footprint_id:', footprint_id)

    fig, ax = show_building(
        location=buildings.loc[[footprint_id]],
        geodatasets={
            'parcels': parcels,
            'buildings_fema': buildings_fema,
            'buildings_nsi': buildings_nsi,
            'buildings_microsoft': buildings_microsoft,
        },
        radius=50,
        return_fig_ax=True,
    )
    # multiparcel_footprints_to_split.loc[[footprint_id]].to_crs('epsg:3857').boundary.plot(
    # ax=ax, color='red'
    # )
    ax.set_title('Example: multi-parcel footprint')

### Infer buildings from parcels
Currently only including footprints - no NSI data

In [ ]:
BUILDING_FROM_PARCELS_COLS = [
    'purpose_group',
    'improvement_value_per_ha',
    'has_duplicate_geometry',
]

N_FOOTPRINTS_MEAN_PER_GROUP_MIN = 0.2

IMPROVEMENT_VALUE_PER_HA_QUANTILE = 0.05

In [ ]:
# Join parcel data to building-parcel links to compute purpose-group
# specific statistics
footprint_parcel_data = footprint_parcels[['link']].join(
    unique_parcel_polygons[~unique_parcel_polygons['has_duplicate_geometry']][
        BUILDING_FROM_PARCELS_COLS
    ],
    on='parcel_id',
    how='inner',
)

In [ ]:
n_footprints_per_group = (
    (
        footprint_parcel_data['purpose_group'].value_counts()
        / unique_parcel_polygons['purpose_group'].value_counts()
    )
    .fillna(0)
    .rename('n_footprints_mean')
)
n_footprints_per_group.sort_values()

In [ ]:
imp_val_quant_column = f'improvement_value_per_ha_q{IMPROVEMENT_VALUE_PER_HA_QUANTILE}'

# Get improvement value for 1-1 matching footprints.
improvement_value_per_ha_q_by_parcel_group = (
    (
        footprint_parcel_data.sample(frac=1)
        .reset_index()
        .drop_duplicates('parcel_id', keep=False)
        .groupby('purpose_group')['improvement_value_per_ha']
        .quantile((IMPROVEMENT_VALUE_PER_HA_QUANTILE, 0.5))
    )
    .unstack()
    .rename(
        columns={
            IMPROVEMENT_VALUE_PER_HA_QUANTILE: imp_val_quant_column,
            0.5: 'improvement_value_per_ha_median',
        }
    )
)

improvement_value_per_ha_q_by_parcel_group.sort_values(
    'improvement_value_per_ha_median'
).map('$ {0:,.0f}'.format)

In [ ]:
mask_parcels_without_footprints = ~unique_parcel_polygons.index.isin(
    footprint_parcels.index.get_level_values('parcel_id').unique()
)

parcel_candidates = (
    unique_parcel_polygons[mask_parcels_without_footprints][
        [
            'purpose_group',
            'improvement_value',
            'improvement_value_per_ha',
            'has_duplicate_geometry',
            'geometry',
        ]
    ]
    .join(improvement_value_per_ha_q_by_parcel_group, on='purpose_group')
    .join(n_footprints_per_group, on='purpose_group')
)

# Lowest improvement value to mark new buildings: low quantile of
# per-area (hectare) improvement value among all parcels with non-zero
# improvement value (thresholds will vary by county).
imp_value_per_ha_min = (
    unique_parcel_polygons[
        unique_parcel_polygons.index.isin(
            footprint_parcels.index.get_level_values('parcel_id').unique()
        )
    ]
    .query('improvement_value_per_ha > 0')['improvement_value_per_ha']
    .quantile(IMPROVEMENT_VALUE_PER_HA_QUANTILE)
)
print(f'Buildings inferred if improvement value > $ {imp_value_per_ha_min:,.0f}.')

mask_parcels_with_inferred_buildings = parcel_candidates['n_footprints_mean'].gt(
    N_FOOTPRINTS_MEAN_PER_GROUP_MIN
) & parcel_candidates['improvement_value_per_ha'].gt(
    parcel_candidates[imp_val_quant_column].div(2).clip(lower=imp_value_per_ha_min)
)

In [ ]:
parcels_with_inferred_buildings = parcel_candidates[
    mask_parcels_with_inferred_buildings
]

buildings_from_parcels = add_openlocationcode_index(
    parcels_with_inferred_buildings[['geometry']], name='footprint_id'
)

# Remove duplicates (NSI points whose OLC didn't link to parcel)
buildings_from_parcels = buildings_from_parcels[
    ~buildings_from_parcels.index.isin(buildings.index)
]

buildings_from_parcels['source'] = 'parcel'
buildings = pd.concat(
    [buildings, buildings_from_parcels[['geometry', 'source']]]
).sort_index()

mask_duplicate_index_values = buildings.index.duplicated(keep=False)
if mask_duplicate_index_values.any():
    warnings.warn(
        f'Created {mask_duplicate_index_values.sum()} duplicate `building_ids`.'
    )
    buildings = make_index_unique(buildings, sort_duplicates_by_area=True)

In [ ]:
resolve_overlapping_polygons(buildings)  # , keep=False)

In [ ]:
buildings = resolve_overlapping_polygons(buildings, keep=False)

## Join NSI

In [ ]:
# Get IDs
nsi_on_footprints = gpd.sjoin(
    buildings_nsi[['geometry']],
    buildings[['geometry']],
    lsuffix='nsi',
)['footprint_id']
nsi_on_footprints

In [ ]:
if nsi_on_footprints.index.duplicated(keep=False).any():
    nsi_on_footprints_duplicate_links = nsi_on_footprints[
        nsi_on_footprints.index.duplicated(keep=False)
    ].sort_index()
    nsi_on_footprints_duplicate_links

    fig, ax = show_building(
        buildings_nsi.loc[[nsi_on_footprints_duplicate_links.index[0]]],
        geodatasets={
            'parcels': unique_parcel_polygons.join(
                parcels.drop(columns=set(unique_parcel_polygons) & set(parcels))
            ),
            # 'buildings_fema': buildings_fema,
            'buildings_nsi': buildings_nsi,
            # 'buildings_microsoft': buildings_microsoft,
        },
        radius=200,
        return_fig_ax=True,
    )
    buildings.loc[nsi_on_footprints_duplicate_links.iloc[:1]].to_crs(
        'epsg:3857'
    ).boundary.plot(ax=ax, color='lime')
    buildings.loc[nsi_on_footprints_duplicate_links.iloc[1:2]].to_crs(
        'epsg:3857'
    ).boundary.plot(ax=ax, color='red')

In [ ]:
# # Identify NSI points that don't intersect with building footprints
# buildings_nsi_add = buildings_nsi[
#     ~buildings_nsi.index.isin(nsi_on_footprints.index.unique())
# ][['geometry', 'building_id_ubid']]
# buildings_nsi_add_parcels = gpd.sjoin(
#     buildings_nsi_add,
#     unique_parcel_polygons[['geometry']],
# )
# buildings_nsi_add_parcels_unique = buildings_nsi_add_parcels
# buildings_nsi_add_parcels['source'] = 'nsi: parcel'

# # Create bounding boxes for rest
# buildings_nsi_add_ubid_bbox = buildings_nsi_add[
#     ~buildings_nsi_add.index.isin(buildings_nsi_add_parcels.index)
# ]
# buildings_nsi_add_ubid_bbox = decode_ubids(
#     buildings_nsi_add_ubid_bbox['building_id_ubid']
# ).join(buildings_nsi_add_ubid_bbox['building_id_ubid'])
# buildings_nsi_add_ubid_bbox.index = pd.Index(
#     buildings_nsi_add_ubid_bbox['building_id_ubid'].str.slice(0, 12),
#     name='footprint_id',
# )
# buildings_nsi_add_parcels['source'] = 'nsi: ubid'
# if buildings_nsi_add_ubid_bbox.index.duplicated().any():
#     raise ValueError(
#         'Created duplicate `openlocationcode` from `building_id_ubid`.'
#     )

# # Remove duplicates (NSI points near centroids of existing footprints)
# buildings_nsi_add = buildings_nsi_add[
#     ~buildings_nsi_add.index.isin(buildings.index)
# ]
# buildings_nsi_add['source'] = 'nsi'
# buildings = pd.concat(
#     [buildings, buildings_nsi_add[['geometry', 'source']]]
# ).sort_index()
# if buildings.index.duplicated().any():
#     raise ValueError('Created duplicate `building_ids`.')

# buildings_nsi_add_parcels

In [ ]:
# nsi_parcel_ids = gpd.sjoin(
#     buildings_nsi[['geometry']],
#     unique_parcel_polygons[['geometry']],
#     lsuffix='nsi',
#     rsuffix='parcel',
#     how='left',
# ).rename(columns={'index_right': 'parcel_id'})['parcel_id']
# nsi_parcel_ids

# Save buildings

In [ ]:
save_parquet(buildings, path(admin_id, 'building-openplaces-2026'))

In [ ]:
share(
    buildings,
    share_path(admin_id, 'building-openplaces-2026'),
    'share/2026/cheer',
    delete_original=False,
)

In [ ]:
assert False, 'This marks the end of a flattened `for` loop.'

---
# Convert to script

*The above line and heading identify the end of the script.*

*Code below this marker will not be included in the converted `.py` script.*

In [ ]:
from openplaces.flow import convert_to_script, test_script

COMMIT = True
# If True, writes `.py` scripts to 'scripts/.../'.
# If False, writes a test version of the script to 'scripts/_test/...'

In [ ]:
convert_to_script(commit=COMMIT)

# Test script

In [ ]:
test_script(*args_list, committed=COMMIT)

# Loop script

In [ ]:
CHEER_ADMIN3_IDS = 'US-NC-BA US-NC-BT US-NC-BL US-NC-BS US-NC-CD US-NC-CE US-NC-CW US-NC-CM US-NC-CN US-NC-CU US-NC-CI US-NC-DE US-NC-DP US-NC-ED US-NC-FR US-NC-GT US-NC-GE US-NC-HL US-NC-HT US-NC-HD US-NC-HO US-NC-HE US-NC-JH US-NC-JN US-NC-LN US-NC-MR US-NC-NA US-NC-NE US-NC-NO US-NC-ON US-NC-PM US-NC-PU US-NC-PD US-NC-PQ US-NC-PI US-NC-RB US-NC-SP US-NC-SC US-NC-TY US-NC-WK US-NC-WR US-NC-WI US-NC-WY US-NC-WO'.split()

In [ ]:
for admin3_id in CHEER_ADMIN3_IDS:
    print(admin3_id)

    args_list_cheer = ['--admin_ids'] + [admin3_id] + ['--verbose']

    print(' '.join(args_list_cheer))
    test_script(*args_list_cheer)

# Examples of data issues
From Brunswick, NC: `US-NC-BS`

## NSI
### Nonsense
Seems to come from HAZUS/NSI-2015

In [ ]:
# Issue: three NSI points that do not exist (circular driveway)
# BUILDING_ID_NSI = 542841649  # Source: HAZUS/NSI-2015

# Example: a manufactured home classified as $0 and "steel"
# BUILDING_ID_NSI = 542714824  # Source: HAZUS/NSI-2015

# Example: looks like manufactured or single-family homes
# NSI says: >4 times multi-family (2 units)
# FEMA (correct): 3 times manufactured
# Parcel: Commercial, $0 building value
BUILDING_ID_NSI = 542464539

In [ ]:
show_building(
    location=buildings_nsi.loc[[BUILDING_ID_NSI]],
    geodatasets={
        'parcels': parcels,
        'buildings_fema': buildings_fema,
        'buildings_nsi': buildings_nsi,
        'buildings_microsoft': buildings_microsoft,
    },
    radius=250,
)

### Sources

In [ ]:
NSI_SOURCE = 'Parcel'
NSI_SOURCE = 'ESRI'
NSI_SOURCE = 'HAZUS/NSI-2015'

building_source_sample = buildings_nsi[buildings_nsi['source'].eq(NSI_SOURCE)].sample()
building_source_sample

In [ ]:
# 4 NSI points, 4 footprints, NSI says Multi-Family
# building_source_sample = buildings_nsi.loc[[542464539]]

In [ ]:
show_building(
    location=building_source_sample,
    geodatasets={
        'parcels': parcels,
        'buildings_fema': buildings_fema,
        'buildings_nsi': buildings_nsi,
        'buildings_microsoft': buildings_microsoft,
    },
    radius=200,
)

## Footprints

In [ ]:
# Largest footprint. On industrial wasteland. Looks like an error
fig, ax = show_building(
    location=buildings_microsoft.sort_values('m2').tail(),
    geodatasets={
        'parcels': parcels,
        # 'buildings_fema': buildings_fema,
        'buildings_nsi': buildings_nsi,
        'buildings_microsoft': buildings_microsoft,
    },
    radius=600,
    return_fig_ax=True,
)

## Parcels

### Diverse purpose groups

In [ ]:
GROUP_COLUMN = 'purpose_group'

N_MAX_GROUPS = 20

BY_AREA = True

if not BY_AREA:
    top_groups = (
        unique_parcel_polygons[GROUP_COLUMN].value_counts().head(N_MAX_GROUPS).index
    )
else:
    top_groups = (
        unique_parcel_polygons.groupby(GROUP_COLUMN)['ha']
        .sum()
        .sort_values(ascending=False)
        .head(N_MAX_GROUPS)
        .index
    )
parcels_of_top_groups = unique_parcels[unique_parcels[GROUP_COLUMN].isin(top_groups)]
fig, ax = plt.subplots(figsize=(10, 10))
parcels_of_top_groups.plot(
    GROUP_COLUMN,
    ax=ax,
    cmap='tab20',
    legend=True,
    legend_kwds={'loc': 'upper left', 'bbox_to_anchor': (1, 1)},
)
# admin3.boundary.plot(ax=ax, color='black', linewidth=0.3)
ax.axis('off')